In [4]:
'''Interacting with Web APIs 
Problem Statement: Analyzing Weather Data from OpenWeatherMap API 
Dataset: Weather data retrieved from OpenWeatherMap API 
Description: The goal is to interact with the OpenWeatherMap API to retrieve weather data 
for a specific location and perform data modeling and visualization to analyze weather 
patterns over time. 
Tasks to Perform: 
1. Register and obtain API key from OpenWeatherMap. 
2. Interact with the OpenWeatherMap API using the API key to retrieve weather data for 
a specific location. 
3. Extract relevant weather attributes such as temperature, humidity, wind speed, and 
precipitation from the API response. 
4. Clean and preprocess the retrieved data, handling missing values or inconsistent 
formats. 
5. Perform data modeling to analyze weather patterns, such as calculating average 
temperature, maximum/minimum values, or trends over time. 
6. Visualize the weather data using appropriate plots, such as line charts, bar plots, or 
scatter plots, to represent temperature changes, precipitation levels, or wind speed 
variations. 
7. Apply data aggregation techniques to summarize weather statistics by specific time 
periods (e.g., daily, monthly, seasonal). 
8. Incorporate geographical information, if available, to create maps or geospatial 
visualizations representing weather patterns across different locations. 
9. Explore and visualize relationships between weather attributes, such as temperature 
and humidity, using correlation plots or heatmaps. '''
# Data Cleaning and Preparation - Customer Churn

# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Import dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

# 2. Explore dataset
print("First 5 Records:")
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

# 3. Handle missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Convert TotalCharges to numeric (if column exists)
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 4. Remove duplicate records
df.drop_duplicates(inplace=True)

# 5. Standardize inconsistent data
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()

# 6. Convert columns to correct data types
if 'SeniorCitizen' in df.columns:
    df['SeniorCitizen'] = df['SeniorCitizen'].astype(int)

if 'tenure' in df.columns:
    df['tenure'] = df['tenure'].astype(int)

# 7. Identify and handle outliers using IQR
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in num_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = df[col].clip(lower, upper)

# 8. Feature Engineering
if 'tenure' in df.columns:
    df['tenure_group'] = pd.cut(
        df['tenure'],
        bins=[-1, 12, 24, 48, float('inf')],
        labels=['new', 'medium', 'long', 'very_long']
    )

# 9. Normalize numerical features
# 10. Split dataset into training and testing sets

# Separate features and target
X = df.drop(['Churn', 'customerID'], axis=1, errors='ignore')
y = df['Churn']

# Convert categorical columns into numerical columns
X = pd.get_dummies(X, drop_first=True)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
scaler = StandardScaler()

scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

scale_cols = [col for col in scale_cols if col in X_train.columns]

X_train.loc[:, scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test.loc[:, scale_cols] = scaler.transform(X_test[scale_cols])

# 11. Export cleaned dataset
df.to_csv("Cleaned_Telecom_Churn.csv", index=False)

# Display results
print("\nData Cleaning Completed Successfully!")

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

print("\nCleaned Dataset:")
print(df.head())

print("\nCleaned dataset saved as Cleaned_Telecom_Churn.csv")

First 5 Records:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies    

C:\Users\Sakshi\AppData\Local\Temp\ipykernel_22644\3031859229.py:93: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.10237124 -0.71174346 -0.79315493 ... -0.30468611 -0.34539184
 -1.07809507]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train.loc[:, scale_cols] = scaler.fit_transform(X_train[scale_cols])
C:\Users\Sakshi\AppData\Local\Temp\ipykernel_22644\3031859229.py:94: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 1.60848343 -0.9966836   0.34660565 ... -1.11880081  0.95719167
  1.60848343]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_test.loc[:, scale_cols] = scaler.transform(X_test[scale_cols])
